
# Paper 3 demo: decentralized observer-specific Ball-DP

This notebook is the end-to-end companion for `paper3_decentralized_ball_dp.tex`.
It demonstrates the three experimental claims that should make the decentralized paper distinct from Paper 2:

1. **Topology-aware observer privacy.** For each attacked node $j$ and observer $A$, compute the transferred sensitivity $\Delta_{A\leftarrow j}(r)$.
2. **Attack alignment.** Run the exact finite-prior MAP attack under the same Gaussian observer model used in the theorem.
3. **Utility/privacy tradeoff.** Compare Ball-local and standard replacement calibration in a lightweight decentralized prototype learner.

The theorem-side release model used below is
\[
Y_A=(H_{A\leftarrow j}\otimes I_p)s_j(z)+\zeta_A,\qquad
\zeta_A\sim\mathcal N(0,\sigma^2 I).
\]
For clipped embedding features, the Ball and standard per-round sensitivities are
\[
\Delta_t^{\rm ball}(r)=\min\{r,2C\},\qquad
\Delta_t^{\rm std}=2C.
\]


In [ ]:

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantbayes.ball_dp.experiments.paper3_decentralized_common import (
    DEFAULT_ORDERS,
    build_transfer_for_observer,
    covariance_time,
    direct_gaussian_rero_bound,
    embedding_radius_report,
    graph_adjacency,
    graph_distances,
    load_embedding_dataset,
    mechanism_noise_for_target_dp,
    mechanism_step_sensitivities,
    metropolis_mixing_matrix,
    observer_accounting_row,
    run_exact_finite_prior_trial,
    run_noisy_prototype_gossip,
)
from quantbayes.ball_dp.decentralized import account_linear_gaussian_observer

plt.rcParams.update({"figure.dpi": 140, "axes.grid": True, "grid.alpha": 0.25})



## 1. Load one embedding dataset

The demo tries to load MNIST embeddings through the official loader path.  If the local environment lacks the vision stack, it falls back to a small synthetic clustered embedding dataset so that the decentralized accounting and attack cells still run.


In [ ]:

args = argparse.Namespace(
    data_root="/content/quantbayes/data",
    embedding_cache_path=None,
    force_recompute_embeddings=False,
    allow_cpu_fallback=True,
    embedding_batch_size=256,
    num_workers=2,
    encoder_model_name="sentence-transformers/all-MiniLM-L6-v2",
    max_length=128,
    hf_cache_dir=None,
)

try:
    data = load_embedding_dataset(args, "MNIST-embeddings")
    X_train = np.asarray(data.X_train, dtype=np.float32)
    y_train = np.asarray(data.y_train, dtype=np.int32)
    X_test = np.asarray(data.X_test, dtype=np.float32)
    y_test = np.asarray(data.y_test, dtype=np.int32)
    dataset_tag = data.spec.tag
    num_classes = int(data.num_classes)
    print(f"Loaded {data.spec.display_name}: train={X_train.shape}, test={X_test.shape}")
except Exception as exc:
    print("Falling back to synthetic clustered embeddings because MNIST loading failed:", repr(exc))
    rng = np.random.default_rng(0)
    dataset_tag = "synthetic_demo"
    num_classes = 10
    feature_dim = 32
    centers = rng.normal(size=(num_classes, feature_dim)).astype(np.float32)
    centers /= np.maximum(np.linalg.norm(centers, axis=1, keepdims=True), 1e-12)
    y_train = rng.integers(0, num_classes, size=4096, dtype=np.int32)
    y_test = rng.integers(0, num_classes, size=1024, dtype=np.int32)
    X_train = centers[y_train] + 0.25 * rng.normal(size=(len(y_train), feature_dim)).astype(np.float32)
    X_test = centers[y_test] + 0.25 * rng.normal(size=(len(y_test), feature_dim)).astype(np.float32)

feature_dim = int(X_train.shape[1])
radius_report = embedding_radius_report(X_train, seed=0)
radius_report



## 2. Build a graph and its transfer matrix

We use a path graph because it makes observer distance visible.  Metropolis mixing gives a nonnegative, doubly-stochastic matrix, so the transferred-sensitivity computation is exact for the diagonal Gaussian observer noise used here.


In [ ]:

num_nodes = 8
rounds = 8
clip_norm = 1.0
noise_std = 8.0
prior_size = 8
radius = float(radius_report["q80"])

A = graph_adjacency("path", num_nodes)
W = metropolis_mixing_matrix(A)
D = graph_distances(A)

print("Mixing matrix W:")
pd.DataFrame(W).round(3)



## 3. Observer-specific Ball vs standard accounting

For a fixed attacked node, compare several observer views.  The same release noise is used for Ball and standard; only the allowed neighbor set changes.  This isolates the effect of policy-local adjacency.


In [ ]:

attacked_node = 0
observer_modes = ["self", "nearest", "farthest", "all"]
rows = []
for mechanism in ["ball", "standard"]:
    for observer_mode in observer_modes:
        rows.append(
            observer_accounting_row(
                dataset_tag=dataset_tag,
                graph="path",
                W=W,
                distances=D,
                rounds=rounds,
                observer_mode=observer_mode,
                attacked_node=attacked_node,
                radius=radius,
                clip_norm=clip_norm,
                noise_std=noise_std,
                feature_dim=feature_dim,
                prior_size=prior_size,
                mechanism=mechanism,
                orders=DEFAULT_ORDERS,
                dp_delta=1e-6,
            )
        )

acct_df = pd.DataFrame(rows)
acct_df[[
    "mechanism", "observer_mode", "observer_nodes", "graph_distance",
    "transferred_sensitivity", "direct_gaussian_rero_bound", "dp_epsilon"
]].round(4)



## 4. Heatmap over attacked and observing nodes

The heatmap below is the decentralized identity of the paper: privacy is not one scalar.  It is directional and observer-specific.


In [ ]:

mat = np.zeros((num_nodes, num_nodes), dtype=float)
for observer in range(num_nodes):
    for attacked in range(num_nodes):
        H = build_transfer_for_observer(W=W, rounds=rounds, observer_nodes=(observer,), attacked_node=attacked)
        cov = covariance_time(H.shape[0], noise_std)
        deltas = mechanism_step_sensitivities(
            mechanism="ball", radius=radius, clip_norm=clip_norm, rounds=rounds
        )
        acct = account_linear_gaussian_observer(
            transfer_matrix=H,
            covariance=cov,
            block_sensitivities=deltas,
            parameter_dim=feature_dim,
            orders=DEFAULT_ORDERS,
            radius=radius,
            dp_delta=1e-6,
            attacked_node=attacked,
            observer=(observer,),
            covariance_mode="kron_eye",
            method="auto",
        )
        mat[observer, attacked] = np.sqrt(acct.sensitivity_sq)

fig, ax = plt.subplots(figsize=(6.2, 5.0))
im = ax.imshow(mat, aspect="auto")
ax.set_xlabel("attacked node j")
ax.set_ylabel("observer node a")
ax.set_title(r"Transferred sensitivity $\Delta_{a\leftarrow j}(r)$")
fig.colorbar(im, ax=ax)
plt.show()



## 5. Radius and noise ablations

The radius $r$ is the policy knob.  Increasing $r$ should increase the Ball bound until clipping dominates.  Increasing $\sigma$ should reduce the bound.


In [ ]:

radii = [radius_report["q50"], radius_report["q80"], radius_report["q95"]]
noises = [4.0, 8.0, 16.0]
ablation_rows = []
for r in radii:
    for sigma in noises:
        for mechanism in ["ball", "standard"]:
            ablation_rows.append(
                observer_accounting_row(
                    dataset_tag=dataset_tag,
                    graph="path",
                    W=W,
                    distances=D,
                    rounds=rounds,
                    observer_mode="farthest",
                    attacked_node=0,
                    radius=float(r),
                    clip_norm=clip_norm,
                    noise_std=float(sigma),
                    feature_dim=feature_dim,
                    prior_size=prior_size,
                    mechanism=mechanism,
                    orders=DEFAULT_ORDERS,
                    dp_delta=1e-6,
                )
            )

abl = pd.DataFrame(ablation_rows)
fig, ax = plt.subplots()
for mechanism, g in abl[abl.noise_std == 8.0].groupby("mechanism"):
    ax.plot(g.radius, g.direct_gaussian_rero_bound, marker="o", label=mechanism)
ax.set_xlabel("radius r")
ax.set_ylabel("direct Gaussian ReRo bound")
ax.set_ylim(0, 1.02)
ax.legend()
plt.show()

fig, ax = plt.subplots()
for mechanism, g in abl[np.isclose(abl.radius, radius)].groupby("mechanism"):
    ax.plot(g.noise_std, g.direct_gaussian_rero_bound, marker="o", label=mechanism)
ax.set_xscale("log")
ax.set_xlabel("noise std sigma")
ax.set_ylabel("direct Gaussian ReRo bound")
ax.set_ylim(0, 1.02)
ax.legend()
plt.show()



## 6. Exact finite-prior MAP attack

Now we audit the release with the statistically correct adversary.  Each trial samples a finite prior inside a radius-$r$ ball, draws the true candidate uniformly, releases the Gaussian observer view, and scores every candidate by the exact Gaussian likelihood.


In [ ]:

rng = np.random.default_rng(123)
attack_rows = []
center_indices = rng.choice(np.arange(len(X_train)), size=20, replace=False)
for t, idx in enumerate(center_indices):
    row = run_exact_finite_prior_trial(
        W=W,
        distances=D,
        rounds=rounds,
        observer_mode="farthest",
        attacked_node=0,
        radius=radius,
        clip_norm=clip_norm,
        noise_std=noise_std,
        prior_size=prior_size,
        feature_dim=feature_dim,
        center=X_train[idx],
        label=int(y_train[idx]),
        rng=np.random.default_rng(10_000 + t),
        orders=DEFAULT_ORDERS,
        dp_delta=1e-6,
    )
    attack_rows.append(row)

attack_df = pd.DataFrame(attack_rows)
print("Empirical exact-ID success:", attack_df.exact_identification_success.mean())
print("Mean direct Gaussian ReRo bound:", attack_df.direct_gaussian_rero_bound.mean())
attack_df[["exact_identification_success", "prior_rank", "mse", "direct_gaussian_rero_bound", "transferred_sensitivity"]].head()



## 7. Utility/privacy tradeoff with a decentralized prototype learner

This is not the main theorem, but it is useful experimentally: calibrate Gaussian noise under Ball-local and standard replacement sensitivities, then run gossip on noisy class-sum prototypes.  Counts are treated as public here; the protected contribution is the feature vector conditional on label.


In [ ]:

epsilon_grid = [2.0, 4.0, 8.0]
utility_rows = []
for mechanism in ["ball", "standard"]:
    for eps in epsilon_grid:
        sensitivity = min(radius, 2 * clip_norm) if mechanism == "ball" else 2 * clip_norm
        cal = mechanism_noise_for_target_dp(
            target_epsilon=eps,
            sensitivity=sensitivity,
            orders=DEFAULT_ORDERS,
            delta=1e-6,
        )
        metrics = run_noisy_prototype_gossip(
            X_train=X_train[:6000],
            y_train=y_train[:6000],
            X_test=X_test[:2000],
            y_test=y_test[:2000],
            W=W,
            rounds=12,
            num_classes=num_classes,
            clip_norm=clip_norm,
            noise_std=cal["noise_std"],
            seed=0,
        )
        utility_rows.append({"mechanism": mechanism, "epsilon": eps, **cal, **metrics})

util = pd.DataFrame(utility_rows)
util[["mechanism", "epsilon", "noise_std", "accuracy_mean", "consensus_state_disagreement"]].round(4)


In [ ]:

fig, ax = plt.subplots()
for mechanism, g in util.groupby("mechanism"):
    ax.plot(g.epsilon, g.accuracy_mean, marker="o", label=mechanism)
ax.set_xlabel("target epsilon")
ax.set_ylabel("nearest-prototype accuracy")
ax.set_ylim(0, 1.02)
ax.legend()
plt.show()



## What to check in official runs

The Paper 3 experiments should not be judged only by whether Ball curves are below standard curves.  The important checks are stricter:

- The transferred-sensitivity heatmaps should vary with graph topology and observer location.
- The direct ReRo bounds should be non-vacuous for at least some realistic $(r,\sigma,T)$ regimes.
- Exact MAP attack success should be compared against the same release model used by the theorem.
- Radius ablations should show the intended policy knob: protection degrades as $r$ grows and approaches the standard clipping ceiling.
- Utility should improve only in the regime where Ball-local sensitivity allows materially smaller calibrated noise than standard replacement adjacency.
